In [1]:
# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

### Build LLM

In [2]:
from langchain_ollama import OllamaLLM

model = OllamaLLM(
    model="qwen3:latest",
)

### Use Text Splitter

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def text_spliiter(data, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
    )

    chunks = splitter.split_documents(data)
    return chunks

### Create the Embedding model

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

def hugginface_embeddings():
    return HuggingFaceEmbeddings(
        model_name = "sentence-transformers/paraphrase-MiniLM-L6-v2"
    )

### Use Retrievers
Vector Store-Backed Retriever

In [5]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./support/companypolicies.txt")
txt_data = loader.load()
txt_data

[Document(metadata={'source': './support/companypolicies.txt'}, page_content="1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we 

In [6]:
chunk_txt =  text_spliiter(txt_data, 200, 20)

In [7]:
from langchain_chroma import Chroma

vectordb = Chroma.from_documents(
          documents = chunk_txt
        , embedding = hugginface_embeddings()
        , collection_metadata = {"hnsw:space": "cosine"}
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1977.15it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
vectordb.similarity_search_with_relevance_scores("email policy", k=5)

[(Document(id='599ce3e4-66a2-4267-b243-5e57789cb134', metadata={'source': './support/companypolicies.txt'}, page_content='3.\tInternet and Email Policy'),
  0.8331340551376343),
 (Document(id='99118c62-a6e5-47da-b826-87309838b40c', metadata={'source': './support/companypolicies.txt'}, page_content='Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and'),
  0.5998820662498474),
 (Document(id='cabdee70-ca67-44c0-926c-caf710c99f45', metadata={'source': './support/companypolicies.txt'}, page_content='Our Internet and Email Policy aims to promote safe, responsible usage of digital communication tools that align with our values and legal obligations. Each employee is expected to understand and'),
  0.5947380661964417),
 (Document(id='3704136e-6d8d-4728-a98d-ea8ddd64c966', metadata={'source': './support/companypolicies.txt'}, page_content='Security:

### Simple similarity search

In [9]:
query = "email policy"
retriever = vectordb.as_retriever()
docs = retriever.invoke(query)
docs

[Document(id='599ce3e4-66a2-4267-b243-5e57789cb134', metadata={'source': './support/companypolicies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(id='99118c62-a6e5-47da-b826-87309838b40c', metadata={'source': './support/companypolicies.txt'}, page_content='Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and'),
 Document(id='cabdee70-ca67-44c0-926c-caf710c99f45', metadata={'source': './support/companypolicies.txt'}, page_content='Our Internet and Email Policy aims to promote safe, responsible usage of digital communication tools that align with our values and legal obligations. Each employee is expected to understand and'),
 Document(id='3704136e-6d8d-4728-a98d-ea8ddd64c966', metadata={'source': './support/companypolicies.txt'}, page_content='Security: Safeguard your login credentials, avoiding the sharing of passwords. Exe

#### Retriver with Args

In [10]:
retriever = vectordb.as_retriever(search_kwargs={"k": 1})
docs = retriever.invoke(query)
docs

[Document(id='599ce3e4-66a2-4267-b243-5e57789cb134', metadata={'source': './support/companypolicies.txt'}, page_content='3.\tInternet and Email Policy')]

### MMR Search (Maximal Marginal Relevance)

MMR in vector stores is a technique used to balance the relevance and diversity of retrieved results. It selects documents that are both highly relevant to the query and minimally similar to previously selected documents. This approach helps to avoid redundancy and ensures a more comprehensive coverage of different aspects of the query.

In [11]:
retriever = vectordb.as_retriever(search_type="mmr")
docs = retriever.invoke(query)
docs

[Document(id='599ce3e4-66a2-4267-b243-5e57789cb134', metadata={'source': './support/companypolicies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(id='99118c62-a6e5-47da-b826-87309838b40c', metadata={'source': './support/companypolicies.txt'}, page_content='Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and'),
 Document(id='ace1addb-bfaf-4601-9440-b080fe4e43b0', metadata={'source': './support/companypolicies.txt'}, page_content='Confidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer data only when encryption is applied. Exercise discretion when discussing'),
 Document(id='bfa6dbe6-a8f3-4521-ba95-1571341f1d7c', metadata={'source': './support/companypolicies.txt'}, page_content='Employee Referrals: We encourage and appreciate employee referrals as they contribut

### Similarity score threshold retrieval

In [12]:
retriever = vectordb.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.4}
)
docs = retriever.invoke(query)
docs

[Document(id='599ce3e4-66a2-4267-b243-5e57789cb134', metadata={'source': './support/companypolicies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(id='99118c62-a6e5-47da-b826-87309838b40c', metadata={'source': './support/companypolicies.txt'}, page_content='Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and'),
 Document(id='cabdee70-ca67-44c0-926c-caf710c99f45', metadata={'source': './support/companypolicies.txt'}, page_content='Our Internet and Email Policy aims to promote safe, responsible usage of digital communication tools that align with our values and legal obligations. Each employee is expected to understand and'),
 Document(id='3704136e-6d8d-4728-a98d-ea8ddd64c966', metadata={'source': './support/companypolicies.txt'}, page_content='Security: Safeguard your login credentials, avoiding the sharing of passwords. Exe

### Multi-Query Retriever

In [13]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("./support/langchain_paper.pdf")
pdf_data = loader.load()
pdf_data[1]

Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': './support/langchain_paper.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}, page_content='LangChain helps us to unlock the ability to harness the \nLLM’s immense potential in tasks such as document analysis, \nchatbot development, code analysis, and countless other \napplications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage barriers through translation, LangChain is ready to \nprovide the tools and programming support you need to do \nwithout it that it is not only difficult but also fresh for you. Its \ncore functionalities encompass: \n1. Context-Aware Capabilities: LangChain facilitates the \ndevelopment of applications that are inherently \ncontext-aware. This means that these applications can \nconnect t

In [14]:
chunks_pdf = text_spliiter(pdf_data, 500, 20)

# VectorDB
ids = vectordb.get()["ids"]
vectordb.delete(ids) # We need to delete existing embeddings from previous documents and then store current document embeddings in.
vectordb = Chroma.from_documents(documents=chunks_pdf, embedding=hugginface_embeddings())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9623.39it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
from langchain_classic.retrievers import MultiQueryRetriever

query = "What does the paper say about langchain?"

retriever = MultiQueryRetriever.from_llm(
    retriever=vectordb.as_retriever(), llm=model
)

docs = retriever.invoke(query)
docs

[Document(id='9ec10ddd-ef44-4d78-b5e0-29f5e8878d72', metadata={'creator': 'Microsoft Word', 'title': 's8329 final', 'source': './support/langchain_paper.pdf', 'creationdate': '2023-12-31T03:50:13+00:00', 'page': 1, 'total_pages': 6, 'moddate': '2023-12-31T03:52:06+00:00', 'producer': 'PyPDF', 'author': 'IEEE', 'page_label': '2'}, page_content='core functionalities encompass: \n1. Context-Aware Capabilities: LangChain facilitates the \ndevelopment of applications that are inherently \ncontext-aware. This means that these applications can \nconnect to a language model and draw from various \nsources of context, such as prompt instructions, a few-\nshot examples, or existing content, to ground their \nresponses effectively. \n2. Reasoning Abilities: LangChain equips applications \nwith the capacity to reason effectively. By relying on a'),
 Document(id='2bb602d4-29a9-41c6-bff4-6bbd298d8c47', metadata={'moddate': '2023-12-31T03:52:06+00:00', 'page': 1, 'producer': 'PyPDF', 'author': 'IEEE'

#### Modern Approach for the Multi query retriver

In [16]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate

query_prompt = ChatPromptTemplate.from_template("""
Generate 4 different rephrasings of the user question.
Return each query on a new line. No explanation.

User question: {question}
""")

loader = PyPDFLoader("./support/langchain_paper.pdf")
pdf_data = loader.load()

chunks_pdf = text_spliiter(pdf_data, 500, 20)

# VectorDB
ids = vectordb.get()["ids"]
vectordb.delete(ids) # We need to delete existing embeddings from previous documents and then store current document embeddings in.
vectordb = Chroma.from_documents(documents=chunks_pdf, embedding=hugginface_embeddings())

retriver = vectordb.as_retriever()


llm_chain = query_prompt | model

llm_chain.invoke("What does the paper say about langchain?")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11104.60it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'What does the research paper discuss regarding LangChain?  \nHow does the study address LangChain?  \nWhat does the article cover about LangChain?  \nWhat does the document explore concerning LangChain?'

In [17]:
from langchain_core.runnables import RunnableLambda

def parse_queries(text: str):
    return [q.strip() for q in text.split("\n") if q.strip()]

def retrive_all(queries):
    all_docs = []

    for q in queries:
        docs = retriever.invoke(q)
        all_docs.extend(docs)

    return all_docs

def deduplicate(docs):
    seen = set()
    unique_docs = []

    for doc in docs:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            unique_docs.append(doc)

    return unique_docs


multimodalretriver = llm_chain | RunnableLambda(parse_queries) | RunnableLambda(retrive_all) | RunnableLambda(deduplicate)


multimodalretriver.invoke("What does the paper say about langchain?")

[Document(id='bb391530-461b-4ee7-909e-597fdb8a289d', metadata={'producer': 'PyPDF', 'page': 1, 'title': 's8329 final', 'source': './support/langchain_paper.pdf', 'total_pages': 6, 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'page_label': '2'}, page_content='LangChain helps us to unlock the ability to harness the \nLLM’s immense potential in tasks such as document analysis, \nchatbot development, code analysis, and countless other \napplications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage barriers through translation, LangChain is ready to \nprovide the tools and programming support you need to do \nwithout it that it is not only difficult but also fresh for you. Its'),
 Document(id='f19ac2e2-ed96-4f8d-b4a5-98c330a3d56f', metadata={'total_pages': 6, 'producer': 'PyPDF', 'title': 's8329 final', 'page_label': '2', 'author': 'IEEE', 

In [18]:
from collections import defaultdict
# Notice in the above method i am not doing any fustion of the output i am simply returning then all after removing duplicates.

# But we can remove the duplicates and also rank them by the docuemts which came up multiple times.

def retrieve_all_rrf(queries):
    all_docs = []

    for q in queries:
        docs = retriever.invoke(q)
        all_docs.append(docs) #not flattening here just appending

    return all_docs

def receiprocal_rank_fusion(results, k = 60):
    fused_scores = defaultdict(float)
    doc_map = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            doc_id = doc.page_content

            fused_scores[doc_id] += 1 / (k + rank + 1) # rank + 1 because rank(index) is 0 based

            doc_map[doc_id] = doc

    reranked_docs = sorted(
        fused_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [doc_map[doc_id] for doc_id, _ in reranked_docs]

multimodalretriver_rrf = llm_chain | RunnableLambda(parse_queries) | RunnableLambda(retrieve_all_rrf) | RunnableLambda(receiprocal_rank_fusion)

multimodalretriver_rrf.invoke("What does the paper say about langchain?")

[Document(id='f19ac2e2-ed96-4f8d-b4a5-98c330a3d56f', metadata={'producer': 'PyPDF', 'moddate': '2023-12-31T03:52:06+00:00', 'author': 'IEEE', 'total_pages': 6, 'title': 's8329 final', 'creator': 'Microsoft Word', 'page_label': '2', 'source': './support/langchain_paper.pdf', 'page': 1, 'creationdate': '2023-12-31T03:50:13+00:00'}, page_content='core functionalities encompass: \n1. Context-Aware Capabilities: LangChain facilitates the \ndevelopment of applications that are inherently \ncontext-aware. This means that these applications can \nconnect to a language model and draw from various \nsources of context, such as prompt instructions, a few-\nshot examples, or existing content, to ground their \nresponses effectively. \n2. Reasoning Abilities: LangChain equips applications \nwith the capacity to reason effectively. By relying on a'),
 Document(id='eed8535b-b229-439c-8236-2cdb77163ad8', metadata={'producer': 'PyPDF', 'moddate': '2023-12-31T03:52:06+00:00', 'creationdate': '2023-12-31

### Self-Querying Retriever

In [19]:
from langchain_core.documents import Document
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from lark import lark

In [20]:
docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]

In [21]:
vectordb = Chroma.from_documents(docs, hugginface_embeddings())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7117.42it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating", description="A 1-10 rating for the movie", type="float"
    ),
]

In [23]:
document_content_description = "Brief summary of a movie."

retriever = SelfQueryRetriever.from_llm(
    model,
    vectordb,
    document_content_description,
    metadata_field_info
)

In [24]:
retriever.invoke("I want to watch a movie rated higher than 8.5")

[Document(id='08e814d4-2f5b-4097-aad8-dbffb2336e1c', metadata={'rating': 8.6, 'director': 'Satoshi Kon', 'year': 2006}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea'),
 Document(id='8a213c9b-9945-45eb-9c5e-ea6db68609ab', metadata={'year': 1979, 'genre': 'thriller', 'director': 'Andrei Tarkovsky', 'rating': 9.9}, page_content='Three men walk into the Zone, three men walk out of the Zone')]

#### Modern Approach for the self query Retriver

In [25]:
from pydantic import BaseModel, Field
from typing import Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableLambda

prompt = ChatPromptTemplate.from_template("""
You are a query parser.

Extract structured information from the user query.

Return ONLY valid JSON in this format:
{{
    "query": string,
    "genre": string or null,
    "year": integer or null,
    "director": string or null,
    "rating": float or null
}}

User query:
{input}
""")

class MovieQuery(BaseModel):
    query: str = Field(description="The Sementic search query")
    genre: Optional[str] = Field(
        default=None,
        description="Genre of the movie"
    )
    year: Optional[int] = Field(
        default=None,
        description="Release year"
    )
    director: Optional[str] = Field(
        default=None,
        description="Director name"
    )
    rating: Optional[float] = Field(
        default=None,
        description="Minimum rating"
    )

parser = PydanticOutputParser(pydantic_object=MovieQuery)

def build_filter(data: MovieQuery):
    filters = {}

    if data.genre:
        filters["genre"] = data.genre

    if data.year:
        filters["year"] = data.year

    if data.director:
        filters["director"] = data.director

    if data.rating:
        # Example: rating >= value
        filters["rating"] = {"$gte": data.rating}

    return filters

def customerRetriver(data: MovieQuery): # This is a selfQueryRetriver implementaion with modern way
    filters = build_filter(data)

    return vectordb.similarity_search(
        query = data.query,
        filter= filters
    )

chain = prompt | model | parser | RunnableLambda(customerRetriver)

In [26]:
chain.invoke("I want to watch a movie rated higher than 8.5")

[Document(id='08e814d4-2f5b-4097-aad8-dbffb2336e1c', metadata={'year': 2006, 'director': 'Satoshi Kon', 'rating': 8.6}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea'),
 Document(id='8a213c9b-9945-45eb-9c5e-ea6db68609ab', metadata={'year': 1979, 'director': 'Andrei Tarkovsky', 'genre': 'thriller', 'rating': 9.9}, page_content='Three men walk into the Zone, three men walk out of the Zone')]

### Parent Document Retriver

In [27]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.stores import InMemoryStore

parent_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=20, separator='\n')
child_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator='\n')

vectordb = Chroma(
    collection_name="split_parents", embedding_function=hugginface_embeddings()
)
#vectordb = Chroma.from_documents(documents=chunks_pdf, embedding=watsonx_embedding())
# The storage layer for the parent documents
store = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=vectordb,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

retriever.add_documents(txt_data)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7984.72it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Created a chunk of size 223, which is longer than the specified 200
Created a chunk of size 274, which is longer than the specified 200
Created a chunk of size 262, which is longer than the specified 200
Created a chunk of size 282, which is longer than the specified 200
Created a chunk of size 262, which is longer than the specified 200
Created a chunk of size 270, which is longer than the specified 200
Created a chunk of size 224, which is longer than the specified 200
Created a chunk of size 325, which is longer than the specified 200
Created a chunk of size 300, which is longer than the specified

In [28]:
len(list(store.yield_keys()))

19

In [29]:
sub_docs = vectordb.similarity_search("smoking policy")

In [30]:
print(sub_docs[0].page_content)

5.	Smoking Policy


In [31]:
retrieved_docs = retriever.invoke("smoking policy")
print(retrieved_docs[0].page_content)

Cost Management: Keep personal phone usage separate from company accounts and reimburse the company for any personal charges on company-issued phones.
Compliance: Adhere to all pertinent laws and regulations concerning mobile phone usage, including those related to data protection and privacy.
Lost or Stolen Devices: Immediately report any lost or stolen mobile devices to the IT department or your supervisor.
Consequences: Non-compliance with this policy may lead to disciplinary actions, including the potential loss of mobile phone privileges.
The Mobile Phone Policy is aimed at promoting the responsible and secure use of mobile devices in line with legal and ethical standards. Every employee is expected to comprehend and abide by these guidelines. Regular reviews of the policy ensure its ongoing alignment with evolving technology and security best practices.
5.	Smoking Policy


#### Modern way for the Parent Document Retriver

In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore
from langchain_core.runnables import RunnableLambda
import uuid

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)

parent_docs = []
child_docs = []

for doc in txt_data:
    parents = parent_splitter.split_documents([doc])

    for parent in parents:
        id = str(uuid.uuid4())

        parent.metadata["parent_id"] = id

        parent_docs.append(parent)

        children = child_splitter.split_documents([parent])

        for child in children:
            child.metadata["parent_id"] = id
            child_docs.append(child)

store = InMemoryStore()

store.mset([(doc.metadata["parent_id"], doc) for doc in parent_docs])

vectordb = Chroma.from_documents(
    documents=child_docs,
    embedding=hugginface_embeddings()
)

retriever = vectordb.as_retriever()

def fetch_parent(children):
    parent_ids= [child.metadata["parent_id"] for child in children]

    parents = store.mget(parent_ids)

    return [p for p in parents if p is not None]

parentdocretriver_chain = retriever | RunnableLambda(fetch_parent)

parentdocretriver_chain.invoke("smoking policy")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8575.61it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Document(metadata={'source': './support/companypolicies.txt', 'parent_id': '8ff8930a-3787-4353-8d1b-c9a75871396d'}, page_content='5.\tSmoking Policy'),
 Document(metadata={'source': './support/companypolicies.txt', 'parent_id': 'a930d8de-b4d6-4444-b134-23735a9d5066'}, page_content='Policy Purpose: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises. This policy is in place to ensure a safe and healthy environment for all employees, visitors, and the general public.\nDesignated Smoking Areas: Smoking is only permitted in designated smoking areas, as marked by appropriate signage. These areas have been chosen to minimize exposure to secondhand smoke and to maintain the overall cleanliness of the premises.\nSmoking Restrictions: Smoking inside company buildings, offices, meeting rooms, and other enclosed spaces is strictly prohibited. This includes electronic cigarettes and vaping devices.\nCompliance with Applicable L